# **Kaggle competition -  Home Credit Risk Prediction - H2O Inference**

This is my first Kaggle competition. While I have worked on datasets through the MIT Professional course, this is the first real world dataset. This dataset is complex and huge, hence I plan to take a systematic step by step approach. Understanding the dataset is of utmost importance for a successful data scientist. A good insight into data will help me make better decisions aboout the aggregation I would like to make and any feature engineering once I have a hanlde of all data and features I have used to achieve best possible results. Hence, I will be taking a slow and incremental change approach. This will not only make tracing changes easier, but also enhance my learning by letting me better understand what step leads to what change.

I have learned a lot from fellow Kagglers who are gracious in sharing their code as well as their knowledge. I have extensively refered to some of the following notebooks for my learning process.

Reference files for this is:
- https://www.kaggle.com/code/dksdms4/lb-0-565-improved-baseline-notebook
- https://www.kaggle.com/code/greysky/home-credit-baseline
- https://www.kaggle.com/code/ravi20076/homecredit-starter-inference-v1
- https://www.kaggle.com/code/peizhengwang/lb-0-57-mod-weight-pure-lgb
- https://www.kaggle.com/code/majiaqi111/metric-s-trick-home-credit-lgb-cat-ensemble

*Code for the voting model borrowed from the first reference


## **Loading necessary libraries**

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing
import polars as pl # parquet file I/O (e.g. pl.read_parquet)
import datetime as dt

# library for metrics
from sklearn.metrics import roc_auc_score

# Importing necessary packages 
import h2o
from h2o.automl import H2OAutoML

# loading library
from sklearn.preprocessing import MinMaxScaler

# library to read and write date related files
import pickle

# library for saving and loading trained models
import joblib

# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# library for garbage collection
import gc  # since the data is huge here, regularly cleaning up will free up memory

# import utility 
import homecreditutility_v4 as hcu #version 21

# library to catch and ignore warnings
import warnings
warnings.filterwarnings("ignore")

## **Data Preparation**

We have data from different sources and they are split into 3 depths.
- Depth 0 has base data and static data from both internal and external sources
- Depth 1 has data from both internal and expternal sources that changes over time. We will need to use aggregate the values to use this
- Depth 2 has similar data as depth 1 and we will need to aggregate the values to use this.


#### **Loading data onto dataframes**

In [ ]:
# declare directories for download path
test_path = "/kaggle/input/home-credit-credit-risk-model-stability/parquet_files/test/"

# dictionary of data paths for train data
test_dict = {
    "base": hcu.DataLoader.get_dataframe(test_path + "test_base.parquet"),
    "depth_0": [
        hcu.DataLoader.get_dataframe(test_path + "test_static_cb_0.parquet"),
        hcu.DataLoader.get_dataframe(test_path + "test_static_0_*.parquet", chunked= True),
        ],
    "depth_1": [
        hcu.DataLoader.get_dataframe(test_path + "test_applprev_1_*.parquet","prev1", 1, True),
        hcu.DataLoader.get_dataframe(test_path + "test_tax_registry_a_1.parquet","tra", 1),
        hcu.DataLoader.get_dataframe(test_path + "test_tax_registry_b_1.parquet","trb", 1),
        hcu.DataLoader.get_dataframe(test_path + "test_tax_registry_c_1.parquet","trc", 1),
        hcu.DataLoader.get_dataframe(test_path + "test_credit_bureau_a_1_*.parquet","cba1", 1, True),
        hcu.DataLoader.get_dataframe(test_path + "test_credit_bureau_b_1.parquet","cbb1", 1),
        hcu.DataLoader.get_dataframe(test_path + "test_other_1.parquet","oth", 1),
        hcu.DataLoader.get_dataframe(test_path + "test_person_1.parquet","per1", 1),
        hcu.DataLoader.get_dataframe(test_path + "test_deposit_1.parquet","dep", 1),
        hcu.DataLoader.get_dataframe(test_path + "test_debitcard_1.parquet","deb", 1),
        ],
    "depth_2": [
        hcu.DataLoader.get_dataframe(test_path + "test_applprev_2.parquet","prev2", 2),
        hcu.DataLoader.get_dataframe(test_path + "test_person_2.parquet","per2", 2),
        hcu.DataLoader.get_dataframe(test_path + "test_credit_bureau_b_2.parquet","cba2", 2),
        hcu.DataLoader.get_dataframe(test_path + "test_credit_bureau_a_2_*.parquet","cbb2", 2, True),
        ]
}

In [ ]:
%%time
col_filename = "/kaggle/input/v7-data-prep-homecreditrisk2024/train_cols_b4_featengg_v7.pkl"
cat_filename = "/kaggle/input/v7-data-prep-homecreditrisk2024/categories_v7.pkl"
test_df = hcu.DataPreprocessor.preprocess_test(test_dict, col_filename, cat_filename)
hcu.MemoryOptimizer.CleanMemory()

In [ ]:
del test_dict
hcu.MemoryOptimizer.CleanMemory()

In [ ]:
test_df.info()

#### **Data Cleaning and Feature Engineering**

In [ ]:
# select same features as in train
# load train_df_columns.pkl
with open('/kaggle/input/v7-data-prep-homecreditrisk2024/final_train_cols_v7.pkl', 'rb') as f:
     df_columns = pickle.load(f)
df_columns.remove('target')

test_df = test_df[df_columns]

In [ ]:
test_df.info()

In [ ]:
# describe() first 5 columns of train_df
test_df.iloc[:, :4].describe().T

### **Scaling using MinMaxScaler**

In [ ]:
with open('/kaggle/input/v7-data-prep-homecreditrisk2024/mmscaler_v7.pkl', 'rb') as f:
  scaler = pickle.load(f)

num_cols = list(test_df.select_dtypes(include = np.number).columns)
num_cols.remove('case_id')
num_cols.remove('WEEK_NUM')

test_df[num_cols] = scaler.transform(test_df[num_cols])

In [ ]:
test_df.info()

In [ ]:
test_df.sample(5)

### **Converting test_df into H2O frame**

In [ ]:
# save train_df in .csv file
test_df.to_csv("./test_df.csv", index=False)

In [ ]:
h2o.init()

In [ ]:
#%%time
test_hf = h2o.import_file("./test_df.csv")

In [ ]:
# obtain the features and target for modelling.
x_feature = list(test_df.columns)
x_feature.remove('case_id')
x_feature.remove('WEEK_NUM')

In [ ]:
hcu.MemoryOptimizer.CleanMemory()
gc.collect()

### **H2O best of family stacked ensemble - second best model Prediction**

In [ ]:
# load the voting classifier model
h2omodel = h2o.load_model("/kaggle/input/v7-h2o-homecreditrisk2024/h2omodels/StackedEnsemble_AllModels_1_AutoML_1_20240421_142412")
h2omodel

### **Batch Predictions**

Code inspiration from: https://www.kaggle.com/code/andreynesterov/home-credit-baseline-training

In [ ]:
def predict_proba_in_batches(model, data, batch_size=100000):
    num_samples = data.nrow
    num_batches = int(np.ceil(num_samples / batch_size))
    probabilities = np.zeros((num_samples,))

    for batch_idx in range(num_batches):
        print(f"Processing batch: {batch_idx+1}/{num_batches}")
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, num_samples)
        batch_h2o = data[start_idx:end_idx, :]
        batch_probs = model.predict(batch_h2o).as_data_frame()['p1'].values  # p1 is the probability of class 1
        probabilities[start_idx:end_idx] = batch_probs
        h2o.remove(batch_h2o)  # Remove batch H2OFrame to free up memory

    return probabilities

In [ ]:
%%time
# predict using the trained gbm model
y_pred = predict_proba_in_batches(h2omodel, test_hf[x_feature])

In [ ]:
y_pred

### **Submission**

In [ ]:
# Ensemble prediction
submission = pd.DataFrame({
    "case_id": test_df["case_id"].to_numpy(),
    "score": y_pred
}).set_index('case_id')
submission.to_csv("./submission.csv")